**Goal:** Show that SHAP attributions for `baseline_nonlocal_vertical` exhibit large
sample-to-sample variability and that common summaries such as mean absolute SHAP values
discard sign information that is structurally meaningful in our kernel-based models.

**Note:** Install SHAP before running: `pip install shap`

In [ ]:
import os, sys, json, warnings
import numpy as np
import torch
import xarray as xr
import proplot as pplt
import matplotlib.ticker as mticker
from math import prod
import shap
sys.path.insert(0,'..')
warnings.filterwarnings('ignore')
pplt.rc.update({'toplabel.weight':'normal','leftlabel.weight':'normal','fontsize':14,'figure.dpi':100})

In [ ]:
with open('../scripts/configs.json','r',encoding='utf-8') as f:
    CONFIGS = json.load(f)
SPLITSDIR  = CONFIGS['filepaths']['splits']
WEIGHTSDIR = CONFIGS['filepaths']['weights']
MODELSDIR  = CONFIGS['filepaths']['models']
MODELS     = CONFIGS['models']
FIELDVARS  = CONFIGS['variables']['field']
LOCALVARS  = CONFIGS['variables']['local']
TARGETVAR  = CONFIGS['variables']['target']
LATRANGE   = CONFIGS['domain']['latrange']
LONRANGE   = CONFIGS['domain']['lonrange']
SEEDS      = CONFIGS['training']['seeds']
SPLIT      = 'test'
MODELNAME  = 'baseline_nonlocal_vertical'
MODELCONFIG = next(m for m in MODELS if m['name']==MODELNAME)
PATCHCONFIG = MODELCONFIG['patch']
USELOCAL    = MODELCONFIG['uselocal']
MAXRADIUS   = max(m['patch']['radius']  for m in MODELS)
MAXTIMELAG  = max(m['patch']['timelag'] for m in MODELS)
NFIELDVARS  = len(FIELDVARS)+1   # +1 for the validity-mask channel appended in collate()
NLOCALVARS  = len(LOCALVARS)

In [ ]:
from scripts.data.classes import PatchDataLoader
from scripts.models.classes import ModelFactory

splitdata  = PatchDataLoader.prepare([SPLIT],FIELDVARS,LOCALVARS,TARGETVAR,SPLITSDIR)
result     = PatchDataLoader.dataloaders(
    splitdata,PATCHCONFIG,USELOCAL,LATRANGE,LONRANGE,
    batchsize=500,workers=0,device='cpu',maxradius=MAXRADIUS,maxtimelag=MAXTIMELAG)
dataset    = result['datasets'][SPLIT]
loader     = result['loaders'][SPLIT]
patchshape = dataset.shape()
nlevs      = patchshape[2]

with xr.open_dataset(f'{SPLITSDIR}/{SPLIT}.h5',engine='h5netcdf') as ds:
    lev = ds['lev'].values.astype(np.float32)

model = ModelFactory.build(MODELNAME,MODELCONFIG,patchshape,NFIELDVARS,NLOCALVARS)
state = torch.load(os.path.join(MODELSDIR,f'{MODELNAME}_{SEEDS[0]}.pth'),map_location='cpu')
model.load_state_dict(state)
model.eval()

print(f'nlevs={nlevs}  |  test samples={len(dataset):,}  |  params={sum(p.numel() for p in model.parameters()):,}')

In [ ]:
# Collect all test samples into a single flat tensor  [fieldpatch.flatten(1) | localvalues]
# Feature layout: [ RH×nlevs | θe×nlevs | θe*×nlevs | mask×nlevs | lf | lhf | shf ]
chunks = []
for batch in loader:
    X = torch.cat([batch['fieldpatch'].flatten(1),batch['localvalues']],dim=1).detach()
    chunks.append(X)
X_all = torch.cat(chunks).float()

# Random subset as background reference; all samples used for explanation
rng   = np.random.default_rng(SEEDS[0])
X_bg  = X_all[rng.choice(len(X_all),500,replace=False)]

print(f'Background: {X_bg.shape}  |  Test: {X_all.shape}')

In [ ]:
class FlatBaselineNN(torch.nn.Module):
    """Wraps BaselineNN to accept a single flat input tensor for SHAP."""
    def __init__(self,model,nfieldvars,patchshape,nlocalvars):
        super().__init__()
        self.model  = model
        self.fpshape = (nfieldvars,*patchshape)
        self.fpsize  = prod(self.fpshape)
    def forward(self,X):
        fp = X[:,:self.fpsize].reshape(-1,*self.fpshape)
        return self.model(fp,X[:,self.fpsize:]).unsqueeze(-1)  # (nbatch,1)

wrapper = FlatBaselineNN(model,NFIELDVARS,patchshape,NLOCALVARS)
wrapper.eval()

# GradientExplainer (integrated-gradients-based) is robust to GELU/Dropout architectures
explainer  = shap.GradientExplainer(wrapper,X_bg)
sv         = explainer.shap_values(X_all)          # list of [(N,nfeatures,1)]
shap_raw   = np.array(sv[0] if isinstance(sv,list) else sv,dtype=np.float32)
if shap_raw.ndim==3: shap_raw = shap_raw[:,:,0]     # (N, nfeatures)

# First 3×nlevs columns correspond to the three atmospheric field variables
shap_fields = shap_raw[:,:3*nlevs].reshape(-1,3,nlevs)  # (N, 3, nlevs)

print(f'SHAP shape: {shap_fields.shape}  [samples × vars × levels]')

In [ ]:
# Load nonparametric kernel weights for the same vertical configuration
with xr.open_dataset(
        os.path.join(WEIGHTSDIR,f'kernel_nonparametric_vertical_{SPLIT}_weights.nc'),
        engine='h5netcdf') as ds:
    k    = ds['k'].load()                          # (field, lev, seed)
    klev = ds['lev'].values.astype(np.float32)

shapmean = shap_fields.mean(axis=0)    # (3, nlevs)
shapstd  = shap_fields.std(axis=0)
kmean    = k.mean('seed').values        # (3, nlevs)
kstd     = k.std('seed').values

# 20 random individual profiles to illustrate sample-to-sample spread
sampleidx = np.random.default_rng(0).choice(len(shap_fields),20,replace=False)

FIELDLABELS = ['RH',r'$\mathit{\theta_e}$',r'$\mathit{\theta_e^*}$']

fig,axs = pplt.subplots(nrows=2,ncols=3,sharey=True,sharex='limits',
                        width=8,height=7,wspace=1,hspace=1.5)
axs.format(rowlabels=['SHAP (baseline)','Kernel weights'],collabels=FIELDLABELS,
           grid=False,ylabel='Pressure (hPa)',ylim=(500,1000),yticks=100,yminorticks='none')

for col in range(3):
    # ── Top row: SHAP mean ± 1σ with individual sample traces ────────────────
    ax = axs[0,col]
    for i in sampleidx:
        ax.plot(shap_fields[i,col],lev,color='blue6',alpha=0.12,linewidth=0.7,zorder=1)
    ax.fill_betweenx(lev,shapmean[col]-shapstd[col],shapmean[col]+shapstd[col],
                     color='blue6',alpha=0.30,label=r'$\pm 1\sigma$',zorder=2)
    ax.plot(shapmean[col],lev,color='blue6',linewidth=1.8,label='Mean',zorder=3)
    ax.axvline(0,color='k',linewidth=0.5)
    ax.set_xlabel('SHAP value',fontsize=11)
    ax.xaxis.set_major_locator(mticker.MaxNLocator(nbins=4,prune='both'))
    ax.xaxis.set_minor_locator(mticker.NullLocator())

    # ── Bottom row: kernel weights mean ± seed σ ─────────────────────────────
    ax2 = axs[1,col]
    ax2.fill_betweenx(klev,kmean[col]-kstd[col],kmean[col]+kstd[col],
                      color='yellow6',alpha=0.25,label=r'$\pm 1\sigma$')
    ax2.plot(kmean[col],klev,color='yellow6',linewidth=1.5,label='Mean')
    ax2.axvline(0,color='k',linewidth=0.5)
    ax2.set_xlabel('Normalized kernel weight',fontsize=11)
    ax2.xaxis.set_major_locator(mticker.MaxNLocator(nbins=3,prune='both'))
    ax2.xaxis.set_minor_locator(mticker.NullLocator())

axs.format(yreverse=True)
axs[0,0].legend(loc='ur',ncols=1,frame=False,prop={'size':11})
axs[1,0].legend(loc='ur',ncols=1,frame=False,prop={'size':11})
pplt.show()
fig.save('../figs/shap_vs_kernels.png',dpi=300)